# MECHA for developers, 5: extend the component

**What you will learn:** add a state variable and a process to `MechaHydraulics` by subclassing it, run the new
component, find it in the declarations and the schedule, and test it.

**You need:** developer tutorial 1 (the plant model of GRANAP and MECHA). **Run time:** about 10 seconds.

## 1. A new variable and its process

Suppose an analysis needs, at each node of the section, **how much lower its water potential is than the soil's** in
the standard problem. A subclass declares it as a state variable and computes it in a step:

- `state_variable` declares it: its unit, its description, its initial value, its scale (`Compartment`, the
  section's nodes) and its location (`node`; an edge's variable would be at `Connection`, `edge`);
- the method `_standard_drop` computes it. Its arguments name the variables it reads, and the array it returns is the
  variable it writes. A parameter is an argument too (`standard_psi_soil`): it comes broadcast to the step's location
  (the nodes, its other arguments'), one value per node, each node its plant's (plants may differ); a step never
  reads a parameter through `self`;
- `@totalstate` places it in the schedule after the solves (`state`), next to `_kr`;
- `@dataclass` is needed on the subclass, as on every component.

In [1]:
from dataclasses import dataclass

import numpy as np
import pandas as pd

from openalea.mecha.component import MechaHydraulics
from openalea.metafspm import state_variable, totalstate
from openalea.metafspm.data_structure.configs import ScalesConfig


@dataclass
class MechaWithDrop(MechaHydraulics):
    standard_drop: float = state_variable(unit="hPa", description="how much lower the node's water potential is than "
                                                                  "the soil's, in the standard problem",
                                          default=0., scale=ScalesConfig.Compartment, location="node",
                                          kind="non_inertial_intensive")

    @totalstate(include_inactive=True)
    def _standard_drop(self, standard_psi, standard_psi_soil) -> np.ndarray:
        return np.asarray(standard_psi_soil, dtype=float) - np.asarray(standard_psi, dtype=float)

## 2. Where it appears

In [2]:
declarations = MechaWithDrop.declarations()
print(declarations[declarations["name"] == "standard_drop"][["name", "kind", "scale", "unit"]].to_string())
print()
for stages, steps in MechaWithDrop.schedule():
    print(f"{', '.join(stages):>12}  {', '.join(steps)}")

              name                    kind        scale unit
183  standard_drop  non_inertial_intensive  Compartment  hPa

        rate  _cell_rank, _conductance, _elongation_sink, _osmotic_drive, _axial_conduit, _xylem_axial_conductance, _phloem_axial_conductance, _collar_node_condition
       state  _standard_problem, _plant_problem
  totalstate  _kr, _standard_transmembrane_fraction, _standard_drop


## 3. Run it

The plant model of developer tutorial 1, with the subclass in place of `MechaHydraulics`:

In [3]:
from openalea.granap.metafspm.architectures import RootAxis
from openalea.granap.metafspm.component import GranapAnatomy, junction_rules
from openalea.metafspm import CompositeModel, Planting, planting, Scene, Translator


class RootHydraulics(CompositeModel):
    initiators = (RootAxis, GranapAnatomy)
    from_scale, nodes, wiring = "SubOrgan", "Compartment", junction_rules

    parts = {"anatomy": GranapAnatomy, "hydraulics": MechaWithDrop}


scene = Scene(planting([Planting()], RootHydraulics))
scene.run()
data_structure = scene.populations[0].data_structure
drop = np.asarray(data_structure.get("standard_drop"))
cells = np.asarray(data_structure.get("compartment_type")) == 0
pd.Series(drop[cells]).groupby(np.asarray(data_structure.get("tissue"), dtype=object)[cells]).mean().sort_values().round(1)

epidermis       13.5
exodermis       54.1
air space      394.4
cortex         529.4
endodermis    2901.8
pericycle     4682.2
phloem        4946.4
stele         4967.2
protoxylem    5000.0
metaxylem     5000.0
dtype: float64

## 4. Test it

A test states what the step must give, on an independent computation. In the package it would go in
`test/component/`, run by pytest; here it runs in place:

In [4]:
def test_the_drop_is_the_soil_minus_the_potential():
    np.testing.assert_allclose(drop, 0. - np.asarray(data_structure.get("standard_psi")), rtol=0, atol=1e-9)
    xylem = np.isin(np.asarray(data_structure.get("cgroup")), (13, 19, 20))
    assert np.allclose(drop[xylem], 5000., atol=1.)                 # the vessels are held at -5000 hPa


test_the_drop_is_the_soil_minus_the_potential()
print("passed")

passed


## 5. Further

- A variable another model writes is declared with `input_variable(..., by="Soil")` (its role). A coupling
  translator, or the same name in the plant model, connects it to the providing model's variable (developer
  tutorials 1 and 4, and metafspm's tutorial).
- A new conductance or boundary term belongs in `conductance.py`, called by `_conductance`; the solves are the
  `graph_system` steps `_standard_problem` and `_plant_problem`.
- `MechaSolutes` and `MechaOsmoticPresets` are components of their own, coupled to `MechaHydraulics` in the same
  way (developer tutorial 3).

**Next:** `06_solute_storage_and_decay.ipynb`, the solutes' storage, decay and liquid boundary.